# UltraGPT: Dedicated Inference Studio & Live Conversational Bot
### Real-Time Streaming Chat, Context Memory Management, Sampling Exploration, and OpenAI API Serving

This notebook is an interactive environment for executing inference on your trained UltraGPT model. It automatically loads your trained weights (`checkpoints/latest_checkpoint.pt`) and character vocabulary without needing to re-run any training steps.

---

### Features:
1. **Model Loader & Checkpoint Telemetry**: Restores trained weights, steps, and loss.
2. **Sampling Exploration**: Side-by-side comparison of Greedy, Top-K, Top-P (Nucleus), and Repetition Penalties.
3. **Live Interactive Bot (Console Mode)**: Type in the prompt box and watch multi-turn responses stream in real time.
4. **Live Interactive Bot (Rich GUI Mode)**: Full visual chat interface with parameter sliders powered by `ipywidgets`.
5. **Context Memory & Sliding Window**: Inspects how dialogue turns are retained and trimmed within `block_size`.
6. **OpenAI API Serving Layer**: In-process OpenAI-compatible FastAPI `/v1/chat/completions` verification.


## 1. Environment & Hardware Detection
Verify GPU device placement and import dependencies.


In [63]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import time
import json
from pathlib import Path
from typing import Any, Dict, Generator, List, Optional, Set, Tuple, Union

# Ensure project root is in path
if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

import torch
import torch.nn as nn
import torch.nn.functional as F

from config import GPTConfig
from model.gpt import GPT
from data.tokenizer import CharTokenizer, BPETokenizer
from data.cleaner import DialogueFormatter
from inference.sampler import sample_next_token
from inference.bot import ConversationBot

# Device placement
device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
print(f"Active Compute Device: {device.upper()}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")


Active Compute Device: CUDA
GPU: Quadro T2000 (4.00 GB VRAM)


## 2. Load Trained Checkpoint & Vocabulary
Restores model weights from `checkpoints/latest_checkpoint.pt` and vocabulary from `checkpoints/char_vocab.json`.


In [64]:
best_ckpt = Path("checkpoints/best_checkpoint.pt")
ckpt_path = best_ckpt if best_ckpt.exists() else Path("checkpoints/latest_checkpoint.pt")
vocab_path = Path("checkpoints/char_vocab.json")

if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}. Please verify training checkpoint exists.")

print(f"Loading checkpoint from: {ckpt_path} ...")
checkpoint = torch.load(ckpt_path, map_location=device)

# Load configuration and initialize architecture
cfg_dict = checkpoint["config"]
config = GPTConfig(**cfg_dict)
model = GPT(config)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

# Load Tokenizer
if vocab_path.exists():
    tokenizer = CharTokenizer.load(vocab_path)
    print(f"Loaded CharTokenizer from: {vocab_path} (Vocab Size: {tokenizer.vocab_size})")
else:
    print("Warning: char_vocab.json not found, falling back to BPE Tokenizer")
    tokenizer = BPETokenizer()

total_params, trainable_params = model.get_num_params()

print("\n" + "=" * 55)
print("UltraGPT Model Successfully Loaded into VRAM")
print("=" * 55)
print(f"  Preset:             {config.preset_name or 'Custom'}")
print(f"  Training Steps:     {checkpoint.get('step', 'N/A'):,}")
print(f"  Validation Loss:    {checkpoint.get('loss', 0.0):.4f}")
print(f"  Total Parameters:   {total_params:,}")
print(f"  Layers / Heads:     {config.n_layer} layers / {config.n_head} heads")
print(f"  Embedding Dim:      {config.n_embd}")
print(f"  Context Window:     {config.block_size} tokens")
print(f"  Device Placement:   {device.upper()}")
print("=" * 55)


Loading checkpoint from: checkpoints\best_checkpoint.pt ...
Loaded CharTokenizer from: checkpoints\char_vocab.json (Vocab Size: 140)

UltraGPT Model Successfully Loaded into VRAM
  Preset:             gpt-nano
  Training Steps:     14,000
  Validation Loss:    2.2739
  Total Parameters:   103,920
  Layers / Heads:     3 layers / 3 heads
  Embedding Dim:      48
  Context Window:     256 tokens
  Device Placement:   CUDA


## 3. Sampling Strategy Exploration
Compare different decoding mechanisms on the exact same prompt:
- **Greedy Decoding** ($T=0$): Deterministic maximum probability selection.
- **Top-K Sampling**: Restricts sampling pool to top $k$ candidates.
- **Nucleus (Top-P) Sampling**: Truncates the tail to keep cumulative mass $p$.
- **Repetition Penalty**: Penalizes previous tokens to prevent repetitive loops.


In [65]:
@torch.no_grad()
def generate_text(
    model: nn.Module,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.8,
    top_k: Optional[int] = 50,
    top_p: Optional[float] = 0.9,
    repetition_penalty: float = 1.1,
) -> str:
    model.eval()
    tokens = tokenizer.encode(prompt)
    current = list(tokens)
    generated = []
    block_size = getattr(model.config, "block_size", 64)

    for _ in range(max_new_tokens):
        ctx = current[-block_size:]
        x = torch.tensor([ctx], dtype=torch.long, device=device)
        logits, _ = model(x)
        next_logits = logits[0, -1, :]

        next_tok = sample_next_token(
            logits=next_logits,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            generated_tokens=generated,
        )

        if next_tok == tokenizer.eos_token_id:
            break

        current.append(next_tok)
        generated.append(next_tok)

    return tokenizer.decode(generated)

test_prompt = "Artificial intelligence is"
print(f"Prompt: '{test_prompt}'\n")

print("--- 1. Greedy Decoding (temp=0.0) ---")
print(test_prompt + generate_text(model, tokenizer, test_prompt, temperature=0.0))

print("\n--- 2. Balanced Nucleus (temp=0.7, top_p=0.9, top_k=40) ---")
print(test_prompt + generate_text(model, tokenizer, test_prompt, temperature=0.7, top_k=40, top_p=0.9, repetition_penalty=1.1))

print("\n--- 3. Creative Sampling (temp=1.0, top_p=0.95, top_k=60) ---")
print(test_prompt + generate_text(model, tokenizer, test_prompt, temperature=1.0, top_k=60, top_p=0.95, repetition_penalty=1.15))


Prompt: 'Artificial intelligence is'

--- 1. Greedy Decoding (temp=0.0) ---
Artificial intelligence is the sequence to the sequence the are the sequence to the sequence to the sequence to the sequence to the sequence the the the the the the trireve is the the the the the the the the the the the the the the the the the the the the the the the the the the th

--- 2. Balanced Nucleus (temp=0.7, top_p=0.9, top_k=40) ---
Artificial intelligence is the token all way rescoure the prediction, seq2seq, and data feed-only and the nambilization, an contain uses for the finding layers.

The sequence to masking of the matrix and variantra model where shaing problems and layer heads for model language dista

--- 3. Creative Sampling (temp=1.0, top_p=0.95, top_k=60) ---
Artificial intelligence is a multiment was are the LN) (195. Conparch]

Due Transhitme to k.: 202

W
Titia curity a model trained weight sheman LLM wayiving chient be ald hig can be predly soce language to may can be 20, an web mether

## 4. Live Interactive Bot (Console Mode)
Run the cell below to launch an interactive conversational session directly in the notebook.
- **Type your message** and press Enter.
- The assistant streams tokens **live in real time**.
- **Context is retained**: Ask follow-up questions and test memory retention!
- Type `:history` to view current context, `:reset` to clear memory, or `:exit` to stop.


In [66]:
def run_live_console_chat(bot: ConversationBot):
    """
    Interactive conversational loop in the notebook:
    - Retains multi-turn dialogue memory.
    - Real-time token streaming.
    - Special commands: :history, :reset, :exit
    """
    print("\n" + "=" * 65)
    print(" UltraGPT Live Interactive Chat (Context-Aware)")
    print(" Commands: ':reset' (clear memory), ':history' (view context), ':exit' (quit)")
    print("=" * 65 + "\n")

    while True:
        try:
            user_msg = input("\nYou > ").strip()
            if not user_msg:
                continue

            # Command: exit
            if user_msg.lower() in (":exit", ":quit", "exit", "quit"):
                print("[Chat session ended. Context preserved in bot instance.]")
                break

            # Command: reset memory
            if user_msg.lower() == ":reset":
                bot.reset()
                print("[Context memory reset! All previous turns cleared.]")
                continue

            # Command: inspect stored context
            if user_msg.lower() == ":history":
                print(f"\n[Current Stored History ({len(bot.messages)} turns)]:")
                for i, m in enumerate(bot.messages):
                    print(f"  [{i}] {m['role'].upper()}: {m['content']}")
                continue

            # Stream generation
            print("\nUltraGPT > ", end="", flush=True)
            for token_piece in bot.stream_generate(
                user_message=user_msg,
                max_tokens=512,
                temperature=0.7,
                top_k=50,
                top_p=0.9,
                repetition_penalty=1.1,
            ):
                print(token_piece, end="", flush=True)
            print()

        except (KeyboardInterrupt, EOFError):
            print("\n[Chat session interrupted]")
            break

# Initialize bot instance
live_bot = ConversationBot(
    model=model,
    tokenizer=tokenizer,
    system_prompt="You are UltraGPT, a helpful and precise AI assistant.",
    device=device,
)

# Start live interactive chat
run_live_console_chat(live_bot)



 UltraGPT Live Interactive Chat (Context-Aware)
 Commands: ':reset' (clear memory), ':history' (view context), ':exit' (quit)


UltraGPT > Atented on with found as the linear provide to predict insearcte is the triel sequence of the output by to memoding. The tokens in compution matrix is the decoder of the demb in language in prevent the processed to the is the operations. Engly in a is that is the not the modules pointion that model and sucess the is the decoder. Mas feed.

As tokens the widen multimes that predices of the pointion of process and with the transformer of the is as is secie. Let difference long with matrix search not with processing the self-attention learning in a 

UltraGPT > The model in from an startem statements was in a non-general process that uning a language seq2seq at standat of token is number is the optimal image programming that biased multiply is is an the possitions a series to mestance. The simalle casess as a remaining of the to distance, inding layer

## 5. Live Interactive Bot (Rich Jupyter GUI Widget)
An interactive graphical interface with parameter sliders (`temperature`, `top_p`, `max_tokens`), live streaming responses, and context retention.


In [67]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

gui_bot = ConversationBot(
    model=model,
    tokenizer=tokenizer,
    system_prompt="You are UltraGPT, an expert AI assistant.",
    device=device,
)

# UI Widgets
output_area = widgets.Output(layout=widgets.Layout(height="350px", overflow="auto", border="1px solid #ddd", padding="10px"))
text_input = widgets.Text(placeholder="Type your message here and press Enter...", layout=widgets.Layout(width="70%"))
send_btn = widgets.Button(description="Send", button_style="primary", icon="paper-plane", layout=widgets.Layout(width="12%"))
clear_btn = widgets.Button(description="Clear Memory", button_style="warning", icon="trash", layout=widgets.Layout(width="15%"))

temp_slider = widgets.FloatSlider(value=0.7, min=0.1, max=1.5, step=0.05, description="Temp:", layout=widgets.Layout(width="28%"))
topp_slider = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="Top-P:", layout=widgets.Layout(width="28%"))
tokens_slider = widgets.IntSlider(value=512, min=64, max=1024, step=32, description="Max Toks:", layout=widgets.Layout(width="28%"))

def render_history():
    with output_area:
        clear_output(wait=True)
        html_content = """<style>
            .msg { margin: 8px 0; padding: 10px 14px; border-radius: 10px; max-width: 80%; font-family: sans-serif; }
            .user { background: #e3f2fd; color: #0d47a1; margin-left: auto; text-align: right; }
            .bot { background: #f5f5f5; color: #212121; margin-right: auto; text-align: left; }
            .system { font-size: 11px; color: #888; text-align: center; margin: 4px 0; }
        </style>"""
        for m in gui_bot.messages:
            if m["role"] == "system":
                html_content += f'<div class="system">System: {m["content"]}</div>'
            elif m["role"] == "user":
                html_content += f'<div class="msg user"><b>You:</b> {m["content"]}</div>'
            else:
                html_content += f'<div class="msg bot"><b>UltraGPT:</b> {m["content"]}</div>'
        display(HTML(html_content))

def on_send_clicked(b):
    user_text = text_input.value.strip()
    if not user_text:
        return
    text_input.value = ""

    # Generate with streaming
    with output_area:
        # Append message and stream
        gen = gui_bot.stream_generate(
            user_message=user_text,
            max_tokens=tokens_slider.value,
            temperature=temp_slider.value,
            top_k=50,
            top_p=topp_slider.value,
            repetition_penalty=1.1,
        )
        for _ in gen:
            render_history()
    render_history()

def on_clear_clicked(b):
    gui_bot.reset()
    render_history()

send_btn.on_click(on_send_clicked)
text_input.on_submit(on_send_clicked)
clear_btn.on_click(on_clear_clicked)

# Render initial view
render_history()

controls_row = widgets.HBox([temp_slider, topp_slider, tokens_slider])
input_row = widgets.HBox([text_input, send_btn, clear_btn])
display(widgets.VBox([output_area, input_row, controls_row]))


C:\Users\USER\AppData\Local\Temp\ipykernel_10424\4033553755.py:65: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_send_clicked)


## 6. Sliding-Window Memory & Context Inspector
Inspect how multi-turn conversation tokens are formatted into the ChatML template and verified against the model's `block_size` context limit.


In [68]:
print(f"Model Max Context Length: {live_bot.block_size} tokens")
print(f"Total Stored Turns:       {len(live_bot.messages)}")

# Format conversation into ChatML template
chatml_prompt = DialogueFormatter.format_conversation(live_bot.messages, add_generation_prompt=True)
tokens_used = len(tokenizer.encode(chatml_prompt))

print(f"Context Tokens Utilized:  {tokens_used} / {live_bot.block_size} tokens ({tokens_used / live_bot.block_size * 100:.1f}%)")
print("\nRaw ChatML Template Sent to Neural Network:")
print("-" * 50)
print(chatml_prompt)
print("-" * 50)


Model Max Context Length: 256 tokens
Total Stored Turns:       5
Context Tokens Utilized:  1211 / 256 tokens (473.0%)

Raw ChatML Template Sent to Neural Network:
--------------------------------------------------
<|im_start|>system
You are UltraGPT, a helpful and precise AI assistant.<|im_end|>
<|im_start|>user
hello<|im_end|>
<|im_start|>assistant
Atented on with found as the linear provide to predict insearcte is the triel sequence of the output by to memoding. The tokens in compution matrix is the decoder of the demb in language in prevent the processed to the is the operations. Engly in a is that is the not the modules pointion that model and sucess the is the decoder. Mas feed.

As tokens the widen multimes that predices of the pointion of process and with the transformer of the is as is secie. Let difference long with matrix search not with processing the self-attention learning in a<|im_end|>
<|im_start|>user
exiy<|im_end|>
<|im_start|>assistant
The model in from an startem sta

## 7. OpenAI-Compatible FastAPI Serving Layer Test
Test the FastAPI server in-process using Starlette's `TestClient` to verify OpenAI format compatibility.


In [69]:
from starlette.testclient import TestClient
from server.app import app, load_model_and_tokenizer

# Initialize server state with loaded checkpoint
load_model_and_tokenizer(checkpoint_path="checkpoints/latest_checkpoint.pt", preset=config.preset_name or "gpt-nano")
client = TestClient(app)

print("--- 1. Testing GET /health ---")
resp_health = client.get("/health")
print(json.dumps(resp_health.json(), indent=2))

print("\n--- 2. Testing GET /v1/models ---")
resp_models = client.get("/v1/models")
print(json.dumps(resp_models.json(), indent=2))

print("\n--- 3. Testing POST /v1/chat/completions ---")
payload = {
    "model": "ultra-gpt",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain transformers in simple terms."}
    ],
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 256,
    "stream": False
}
resp_chat = client.post("/v1/chat/completions", json=payload)
chat_data = resp_chat.json()
print("Status Code:", resp_chat.status_code)
print("Response Message:")
print("  Role:   ", chat_data["choices"][0]["message"]["role"])
print("  Content:", chat_data["choices"][0]["message"]["content"])
print("  Usage:  ", chat_data["usage"])
print("\n>>> Inference Notebook Complete: All Features Verified Successfully! <<<")


--- 1. Testing GET /health ---
{
  "status": "healthy",
  "device": "cuda",
  "cuda_available": true,
  "preset": "gpt-nano",
  "vocab_size": 140,
  "block_size": 256,
  "parameters": {
    "total": 103920,
    "trainable": 103920
  }
}

--- 2. Testing GET /v1/models ---
{
  "object": "list",
  "data": [
    {
      "id": "ultra-gpt",
      "object": "model",
      "created": 1794626106,
      "owned_by": "ultra-gpt-team",
      "permission": [],
      "root": "ultra-gpt",
      "parent": null,
      "metadata": {
        "preset": "gpt-nano",
        "total_parameters": 103920
      }
    }
  ]
}

--- 3. Testing POST /v1/chat/completions ---
Status Code: 200
Response Message:
  Role:    assistant
  Content: Acslbcpcp_ec
rp_lqdmpkcp
ck`cbgagle
_jempgrfk
rfc
glqcptcjw
npm`jckq
dpmk
_
rp_lqdmpkcp,`_qcb
ml
rp_lqdmpkcp
kmbcjq+
rfgq
k_rfck_rga_j
ugrf
rp_lqdmpkcp
ufgaf
_jempgrfk-		Sfc
amlc
rp_lqj_rgml
afgacq
_jjmucb
rm
amknjcv
a_l
bck`cb
qcosclac-
Hd
rfc
jc_plcb
qco
  Usage:   {'prompt_token